In [1]:
import pandas as pd
df=pd.read_csv("../data/products.csv")
print("Dataset shape:",df.shape)
display(df.head())


Dataset shape: (15, 5)


,product_id,product_name,category,description,price
0,1,HP Laptop,Electronics,laptop computer for coding and programming,55000
1,2,Dell Laptop,Electronics,laptop computer for students and office work,48000
2,3,Wireless Mouse,Accessories,wireless mouse for laptop and computer,800
3,4,Mechanical Keyboard,Accessories,keyboard for coding and gaming,2500
4,5,Gaming Headphones,Accessories,headphones for gaming and music,3000


In [2]:
import sys
print(sys.executable)

c:\Users\indra\OneDrive\Desktop\e-commerce_recommendation_system\.venv\Scripts\python.exe


In [3]:
import pandas as pd
print(pd.__version__)

2.3.3


In [6]:
print(df.info)
print("\nMissing values:")
print(df.isnull().sum())


<bound method DataFrame.info of     product_id         product_name     category  \
0            1            HP Laptop  Electronics   
1            2          Dell Laptop  Electronics   
2            3       Wireless Mouse  Accessories   
3            4  Mechanical Keyboard  Accessories   
4            5    Gaming Headphones  Accessories   
5            6           Smartphone  Electronics   
6            7               Tablet  Electronics   
7            8         Office Chair    Furniture   
8            9           Study Desk    Furniture   
9           10            USB-C Hub  Accessories   
10          11    Bluetooth Speaker  Electronics   
11          12         Laptop Stand  Accessories   
12          13        Running Shoes      Fashion   
13          14          Smart Watch  Electronics   
14          15         External SSD  Electronics   

                                          description  price  
0          laptop computer for coding and programming  55000  
1        

In [7]:
print("\nProducts per category:")
print(df["category"].value_counts())


Products per category:
category
Electronics    7
Accessories    5
Furniture      2
Fashion        1
Name: count, dtype: int64


In [8]:
df["content"]=(
    df["product_name"]+" "+df["category"]+" "+df["description"]
    
)
display(df[["product_name","content"]].head())

,product_name,content
0,HP Laptop,HP Laptop Electronics laptop computer for cod...
1,Dell Laptop,Dell Laptop Electronics laptop computer for s...
2,Wireless Mouse,Wireless Mouse Accessories wireless mouse for...
3,Mechanical Keyboard,Mechanical Keyboard Accessories keyboard for ...
4,Gaming Headphones,Gaming Headphones Accessories headphones for ...


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
#Convert product text into numerical vectors
tfidf=TfidfVectorizer(stop_words="english")
tfidf_matrix=tfidf.fit_transform(df["content"])
print("TF-IDF matrix shape:",tfidf_matrix.shape)
print("Number of unique words:",len(tfidf.get_feature_names_out()))

TF-IDF matrix shape: (15, 62)
Number of unique words: 62


In [10]:
feature_names=tfidf.get_feature_names_out()
print("First 20 words:")
print(feature_names[:20])
print("\nFirst product vector:")
print(tfidf_matrix[0].toarray())

First 20 words:
['accessories' 'adapter' 'adjustable' 'bluetooth' 'chair' 'coding'
 'comfortable' 'communication' 'computer' 'computers' 'connectivity'
 'dell' 'desk' 'device' 'drive' 'electronics' 'entertainment' 'ergonomic'
 'exercise' 'external']

First product vector:
[[0.         0.         0.         0.         0.         0.38585785
  0.         0.         0.344345   0.         0.         0.
  0.         0.         0.         0.24432307 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.44436693 0.         0.         0.5271836
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.44436693 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]]


In [11]:
from sklearn.metrics.pairwise import cosine_similarity
#compare every product with every other product
similarity_matrix=cosine_similarity(tfidf_matrix)
print("Similarity matrix shape:",similarity_matrix.shape)

Similarity matrix shape: (15, 15)


In [12]:
similarity_df=pd.DataFrame(
    similarity_matrix,
    index=df["product_name"],
    columns=df["product_name"]
)
display(similarity_df.round(2))

product_name,HP Laptop,Dell Laptop,Wireless Mouse,Mechanical Keyboard,Gaming Headphones,Smartphone,Tablet,Office Chair,Study Desk,USB-C Hub,Bluetooth Speaker,Laptop Stand,Running Shoes,Smart Watch,External SSD
product_name,,,,,,,,,,,,,,,
HP Laptop,1.00,0.43,0.19,0.13,0.00,0.06,0.06,0.00,0.12,0.10,0.05,0.21,0.0,0.05,0.04
Dell Laptop,0.43,1.00,0.18,0.00,0.00,0.05,0.05,0.29,0.11,0.09,0.05,0.20,0.0,0.05,0.04
Wireless Mouse,0.19,0.18,1.00,0.05,0.05,0.00,0.00,0.00,0.04,0.08,0.00,0.12,0.0,0.00,0.00
Mechanical Keyboard,0.13,0.00,0.05,1.00,0.26,0.00,0.00,0.00,0.00,0.05,0.00,0.05,0.0,0.00,0.00
Gaming Headphones,0.00,0.00,0.05,0.26,1.00,0.00,0.00,0.00,0.00,0.04,0.10,0.05,0.0,0.00,0.00
Smartphone,0.06,0.05,0.00,0.00,0.00,1.00,0.05,0.00,0.00,0.00,0.05,0.00,0.0,0.05,0.04
Tablet,0.06,0.05,0.00,0.00,0.00,0.05,1.00,0.00,0.00,0.00,0.27,0.00,0.0,0.05,0.04
Office Chair,0.00,0.29,0.00,0.00,0.00,0.00,0.00,1.00,0.18,0.00,0.00,0.00,0.0,0.00,0.00
Study Desk,0.12,0.11,0.04,0.00,0.00,0.00,0.00,0.18,1.00,0.04,0.00,0.29,0.0,0.00,0.00


In [13]:
product_name="HP Laptop"
product_index=df.index[
    df["product_name"]==product_name
][0]
scores=list(enumerate(similarity_matrix[product_index]))
scores=sorted(scores,
              key=lambda x: x[1],
              reverse=True)
for index, score in scores[1:6]:
    print(df.iloc[index]["product_name"],round(score,3))

Dell Laptop 0.426
Laptop Stand 0.214
Wireless Mouse 0.189
Mechanical Keyboard 0.127
Study Desk 0.119


In [14]:
def recommend_products(product_name,number_of_recommendations=5):
    #find the selected product
    matching_products=df.index[
        df["product_name"].str.lower()==product_name.lower()
    ]
    if len(matching_products)==0:
        return f"Product {product_name} was not found."
    product_index=matching_products[0]
    #get similarity scores against all products
    scores=list(enumerate(similarity_matrix[product_index]))
    #Sort from highest similarity to lowest
    scores=sorted(
        scores,
        key=lambda item: item[1],
        reverse=True
    )
    #Exclude the selected product itself
    recommendations=[]
    for index,score in scores:
        if index!=product_index:
            recommendations.append({
                "product_name":df.iloc[index]["product_name"],
                "category":df.iloc[index]["category"],
                "price":df.iloc[index]["price"],
                "similarity_score": round(float(score),3)
            })
        if len(recommendations)==number_of_recommendations:
            break
    return pd.DataFrame(recommendations)

In [15]:
recommend_products("HP Laptop")

,product_name,category,price,similarity_score
0,Dell Laptop,Electronics,48000,0.426
1,Laptop Stand,Accessories,1200,0.214
2,Wireless Mouse,Accessories,800,0.189
3,Mechanical Keyboard,Accessories,2500,0.127
4,Study Desk,Furniture,5000,0.119


In [20]:
from sklearn.metrics.pairwise import cosine_similarity
def search_products(search_text, number_of_recommendations=5):
    #convert the user's search into TF-IDF features
    search_vector=tfidf.transform([search_text])
    #compare the search against every product
    scores=cosine_similarity(
        search_vector,
        tfidf_matrix
    ).flatten()
    #sort products from highest score to lowest
    ranked_indices=scores.argsort()[::-1]
    #select the top matching products
    results=df.iloc[ranked_indices].copy()
    results["similarity_score"]=scores[ranked_indices]
    #return the requested number of products
    results=results[results["similarity_score"]>0]
    return results[
        ["product_name","category","price","similarity_score"]
    ].head(number_of_recommendations).reset_index(drop=True)

In [21]:
search_products(
    "I need accessories for coding and working on a laptop",
    5
)

,product_name,category,price,similarity_score
0,HP Laptop,Electronics,55000,0.408087
1,Study Desk,Furniture,5000,0.324790
2,Mechanical Keyboard,Accessories,2500,0.279626
3,Laptop Stand,Accessories,1200,0.240412
4,Dell Laptop,Electronics,48000,0.183800


In [22]:
search_products(
    "I want a device for fitness tracking",
    3
)

,product_name,category,price,similarity_score
0,Smart Watch,Electronics,6500,0.459960
1,Tablet,Electronics,18000,0.239519
